# Inflation-guard pipeline, live check
`inflation_guard_pipeline.py` (repo root, moved there on 2026-11-15) (pre-registered rule in `results/preregistered_guard.json`). Live (GPU) on 12 official photos that cover every stage (bar, agreement, escalation, skip).
Checks: (1) the guard is silent on clean data (factor 1.0), so stage, place and query count equal the replay of AgreementQueryPipeline on the stored dump; (2) the far-candidate level computed live from the queries equals the level computed offline from the signal table (same 6 standard queries).

In [ ]:
import os, sys, time
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "tests/Test_Iteration_7/changed_src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from train.checks import replay
import train.checks.scale_robust as sr
T7 = ROOT / "tests/Test_Iteration_7/results"
replayed = {pid: replay.decide_current(ph) for pid, ph in replay.load_set(T7 / "rich_dump/official150").items()}
by_stage = {}
for pid, r in replayed.items():
    by_stage.setdefault(r["stage"], []).append(pid)
photo_ids = by_stage["bar"][:4] + by_stage["agreement"][:3] + by_stage["escalation"][:3] + by_stage["skip"][:2]
print({k: len(v) for k, v in by_stage.items()}, "->", photo_ids)
raw = pd.read_csv(T7 / "signals_ours_official150.csv.gz")
keys = {(round(h, 2), round(s, 2)) for h, s in replay.STANDARD}
std = raw[[(round(h, 2), round(s, 2)) in keys for h, s in zip(raw.heading, raw.scale)]]
offline_level = sr.far_levels(std).set_index("photo").far_inl

In [ ]:
from inflation_guard_pipeline import InflationGuardPipeline
pipeline = InflationGuardPipeline("official", "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                  matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                                  map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None)
pipeline.setup(500, 7)
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set

In [ ]:
rows = []
for pid in photo_ids:
    t0 = time.time()
    r = pipeline.run(test_set.items[pid], test_set.priors[pid])
    err = float(np.hypot(r["x"] - test_set.truths[pid][0], r["y"] - test_set.truths[pid][1]) / test_set.sigmas[pid])
    o = replayed[pid]
    rows.append(dict(photo=pid, live_stage=r["stage"], replay_stage=o["stage"], live_queries=r["n_queries_run"], replay_queries=o["n_queries"],
                     live_err_sigma=round(err, 2), replay_err_sigma=round(float(o["err_sigma"]), 2), factor=r["guard_factor"],
                     live_level=round(r["far_level"], 3), offline_level=round(float(offline_level.loc[pid]), 3), seconds=round(time.time() - t0, 1)))
live = pd.DataFrame(rows)
live["same_stage"] = live.live_stage == live.replay_stage
live["same_place"] = (live.live_err_sigma - live.replay_err_sigma).abs() < 0.2
live["same_level"] = (live.live_level - live.offline_level).abs() < 0.02
live

In [ ]:
print("same stage:", int(live.same_stage.sum()), "of", len(live), "| same place:", int(live.same_place.sum()), "| same level:", int(live.same_level.sum()),
      "| factors:", sorted(set(live.factor)))
live.to_csv(T7 / "inflation_guard_live_check.csv", index=False)